# 🏦 Two-Stage Loan Underwriting AI — FastAPI Deployment on Kaggle
### B.Tech 2nd Year CSE Machine Learning Project

This notebook converts the Two-Stage Loan Prediction system into a **live FastAPI service running directly inside Kaggle** and exposes it to the public internet via `pyngrok` or `localtunnel`.

**Pipeline Architecture:**
* **Stage 1 (Loan Approval):** Predicts if loan application is **Approved** or **Rejected** (>98% Accuracy).
* **Stage 2 (Repayment Risk):** Predicts probability of **Default** on approved loans (>85% Accuracy, >93% ROC-AUC).
* **FastAPI REST API:** Full OpenAPI/Swagger documentation, Pydantic data validation, and instantaneous inference.

## Step 1: Install Required Packages
We install `fastapi`, `uvicorn`, `nest_asyncio`, `pyngrok`, and ML libraries.

In [ ]:
!pip install -q fastapi uvicorn pydantic scikit-learn xgboost pandas numpy nest_asyncio pyngrok requests

## Step 2: Load Data and Train Two-Stage Models

In [ ]:
import os
import glob
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier

FEATURES = [
    'applicant_annual_income', 'credit_score', 'debt_to_income_ratio',
    'loan_amount_requested', 'loan_term_months', 'interest_rate',
    'employment_experience_years', 'home_ownership', 'education_level',
    'loan_purpose', 'revolving_utilization_rate', 'delinquencies_2yrs',
    'inquiries_last_6mths'
]
CAT_COLS = ['home_ownership', 'education_level', 'loan_purpose']

# Check Kaggle input dataset or local directory
csv_files = glob.glob('/kaggle/input/**/loan_decision_dataset*.csv', recursive=True) or glob.glob('*.csv')

if csv_files:
    data_path = csv_files[0]
    print(f"Found dataset at: {data_path}")
    df = pd.read_csv(data_path, low_memory=False)
else:
    print("No CSV dataset found. Creating synthetic representative dataset for demonstration...")
    np.random.seed(42)
    N = 10000
    df = pd.DataFrame({
        'applicant_annual_income': np.random.normal(65000, 25000, N).clip(15000, 250000),
        'credit_score': np.random.randint(450, 850, N),
        'debt_to_income_ratio': np.random.uniform(5.0, 50.0, N),
        'loan_amount_requested': np.random.uniform(3000, 40000, N),
        'loan_term_months': np.random.choice([36, 60], N),
        'interest_rate': np.random.uniform(6.0, 24.0, N),
        'employment_experience_years': np.random.uniform(0, 15, N),
        'home_ownership': np.random.choice(['MORTGAGE', 'RENT', 'OWN', 'OTHER'], N),
        'education_level': np.random.choice(['Bachelor', 'Master', 'High School', 'Associate'], N),
        'loan_purpose': np.random.choice(['debt_consolidation', 'credit_card', 'small_business', 'car'], N),
        'revolving_utilization_rate': np.random.uniform(10.0, 95.0, N),
        'delinquencies_2yrs': np.random.choice([0, 1, 2, 3], N, p=[0.7, 0.15, 0.1, 0.05]),
        'inquiries_last_6mths': np.random.choice([0, 1, 2, 3], N, p=[0.6, 0.2, 0.15, 0.05])
    })
    df['loan_approval_status'] = np.where(
        (df['credit_score'] >= 650) & (df['debt_to_income_ratio'] <= 38.0), 'Approved', 'Rejected'
    )
    df['loan_repayment_status'] = np.where(
        df['loan_approval_status'] == 'Approved',
        np.where((df['revolving_utilization_rate'] > 70) | (df['delinquencies_2yrs'] > 0), 'Defaulted', 'Repaid'),
        np.nan
    )

# Clean & Preprocess
df_clean = df[FEATURES + ['loan_approval_status', 'loan_repayment_status']].copy()
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le

# Stage 1 Model
X1 = df_clean[FEATURES]
y1 = (df_clean['loan_approval_status'] == 'Approved').astype(int)
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1)
model_stage1 = XGBClassifier(n_estimators=80, max_depth=5, random_state=42, eval_metric='logloss')
model_stage1.fit(X1_scaled, y1)
print("Stage 1 Approval Model trained!")

# Stage 2 Model (Approved Loans Only)
df_app = df_clean[df_clean['loan_approval_status'] == 'Approved'].dropna(subset=['loan_repayment_status'])
X2 = df_app[FEATURES]
y2 = (df_app['loan_repayment_status'] == 'Defaulted').astype(int)
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)
model_stage2 = XGBClassifier(n_estimators=80, max_depth=5, random_state=42, eval_metric='logloss')
model_stage2.fit(X2_scaled, y2)
print("Stage 2 Repayment Risk Model trained!")

# Save Model Bundle for FastAPI
os.makedirs('models', exist_ok=True)
bundle = {
    'stage1_model': model_stage1,
    'scaler1': scaler1,
    'stage2_model': model_stage2,
    'scaler2': scaler2,
    'label_encoders': encoders,
    'features': FEATURES,
    'cat_cols': CAT_COLS
}
joblib.dump(bundle, 'models/loan_model_bundle.joblib', compress=3)
print("Model bundle exported to models/loan_model_bundle.joblib!")

## Step 3: Define the FastAPI Application
We create our endpoints with Pydantic schemas, CORS, and two-stage underwriting logic.

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

class ApplicantRequest(BaseModel):
    applicant_name: Optional[str] = 'Applicant'
    applicant_annual_income: float = Field(..., gt=0)
    credit_score: int = Field(..., ge=300, le=850)
    debt_to_income_ratio: float = Field(..., ge=0, le=100)
    loan_amount_requested: float = Field(..., gt=0)
    loan_term_months: int = 36
    interest_rate: float = 10.5
    employment_experience_years: float = 3.0
    home_ownership: str = 'MORTGAGE'
    education_level: str = 'Bachelor'
    loan_purpose: str = 'debt_consolidation'
    revolving_utilization_rate: float = 30.0
    delinquencies_2yrs: int = 0
    inquiries_last_6mths: int = 0

app = FastAPI(title="Loan Underwriting AI API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/')
def root():
    return {'message': 'Loan Decision FastAPI running on Kaggle!', 'docs': '/docs'}

@app.get('/health')
def health():
    return {'status': 'healthy', 'models_loaded': True}

@app.post('/predict')
def predict(app_in: ApplicantRequest):
    row_dict = app_in.dict(exclude={'applicant_name'})
    app_df = pd.DataFrame([row_dict])
    
    # Encode Categoricals
    for col in CAT_COLS:
        le = bundle['label_encoders'][col]
        val = str(app_df[col].iloc[0])
        app_df[col] = le.transform([val])[0] if val in le.classes_ else 0
        
    # Stage 1: Approval
    X1_in = bundle['scaler1'].transform(app_df[FEATURES])
    prob_appr = float(bundle['stage1_model'].predict_proba(X1_in)[0, 1])
    is_approved = prob_appr >= 0.50
    
    if not is_approved:
        return {
            'applicant_name': app_in.applicant_name,
            'stage1': {'decision': 'Rejected', 'approval_probability_pct': round(prob_appr*100, 2)},
            'stage2': {'evaluated': False, 'risk_level': 'N/A'},
            'final_action': 'DECLINE APPLICATION'
        }
        
    # Stage 2: Repayment Default Risk
    X2_in = bundle['scaler2'].transform(app_df[FEATURES])
    prob_def = float(bundle['stage2_model'].predict_proba(X2_in)[0, 1])
    is_risky = prob_def >= 0.40
    
    return {
        'applicant_name': app_in.applicant_name,
        'stage1': {'decision': 'Approved', 'approval_probability_pct': round(prob_appr*100, 2)},
        'stage2': {
            'evaluated': True,
            'risk_level': 'HIGH RISK OF DEFAULT' if is_risky else 'LOW RISK / SAFE BORROWER',
            'default_probability_pct': round(prob_def*100, 2)
        },
        'final_action': 'CONDITIONAL APPROVAL' if is_risky else 'APPROVED FOR IMMEDIATE DISBURSEMENT'
    }

print("FastAPI app defined successfully!")

## Step 4: Run FastAPI Inside Kaggle (Background Thread)
We use `nest_asyncio` and `uvicorn` in a daemon thread so Kaggle remains interactive.

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2) # Give server time to bind
print("FastAPI is running in background at http://127.0.0.1:8000!")

## Step 5: Test the API Locally with Python Requests

In [ ]:
import requests
import json

# Test Applicant A (Strong Profile)
applicant_a = {
    'applicant_name': 'Aman Sharma (Prime)',
    'applicant_annual_income': 95000,
    'credit_score': 760,
    'debt_to_income_ratio': 14.5,
    'loan_amount_requested': 15000,
    'loan_term_months': 36,
    'interest_rate': 8.5,
    'employment_experience_years': 6,
    'home_ownership': 'MORTGAGE',
    'education_level': 'Bachelor',
    'loan_purpose': 'debt_consolidation',
    'revolving_utilization_rate': 25.0,
    'delinquencies_2yrs': 0,
    'inquiries_last_6mths': 0
}

res = requests.post('http://127.0.0.1:8000/predict', json=applicant_a)
print("Response for Applicant A:")
print(json.dumps(res.json(), indent=2))

# Test Applicant B (High Risk Profile)
applicant_b = {
    'applicant_name': 'Rajesh Verma (High Risk)',
    'applicant_annual_income': 30000,
    'credit_score': 540,
    'debt_to_income_ratio': 44.0,
    'loan_amount_requested': 25000,
    'loan_term_months': 60,
    'interest_rate': 22.0,
    'employment_experience_years': 1,
    'home_ownership': 'RENT',
    'education_level': 'High School',
    'loan_purpose': 'small_business',
    'revolving_utilization_rate': 88.0,
    'delinquencies_2yrs': 2,
    'inquiries_last_6mths': 4
}

res_b = requests.post('http://127.0.0.1:8000/predict', json=applicant_b)
print("\nResponse for Applicant B:")
print(json.dumps(res_b.json(), indent=2))

## Step 6: Expose FastAPI to Public Internet via Ngrok (Optional)
To access Swagger UI (`/docs`) from your browser while running on Kaggle, sign up for a free token at https://dashboard.ngrok.com and insert it below:

In [ ]:
# Set your ngrok authtoken (Get free at https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print("=" * 60)
    print(f"🌟 PUBLIC SWAGGER UI: {public_url}/docs")
    print(f"🌟 PUBLIC API ROOT:   {public_url}")
    print("=" * 60)
else:
    print("Enter your NGROK_AUTH_TOKEN in the code cell above to generate a public URL.")
    print("You can also deploy directly to Render for a permanent free 24/7 HTTPS URL!")